In [1]:
import librosa
import soundfile as sf
import pandas as pd
import os
from tqdm import tqdm

In [2]:
THRESHOLD = 100
ANNOTATIONS_PATH = "../annotations.csv"
AUDIO_PATH = "../soundscape_data"
OUTPUT_PATH = "../sliced_clips"

os.makedirs(OUTPUT_PATH, exist_ok=True)

ann = pd.read_csv(ANNOTATIONS_PATH)
species_counts = ann['Species eBird Code'].value_counts()
valid_species = species_counts[species_counts >= THRESHOLD].index.tolist()
ann_filtered = ann[ann['Species eBird Code'].isin(valid_species)].reset_index(drop=True)

print(f'Total annotations to slice: {len(ann_filtered)}')
print(f'Species count: {ann_filtered["Species eBird Code"].nunique()}')

Total annotations to slice: 52456
Species count: 42


In [5]:
print('????' in valid_species)

True


In [6]:
valid_species = [s for s in valid_species if s != '????']
ann_filtered = ann[ann['Species eBird Code'].isin(valid_species)].reset_index(drop=True)
print(ann_filtered['Species eBird Code'].nunique())

41


In [7]:
rows = []

for idx, row in tqdm(ann_filtered.iterrows(), total=len(ann_filtered)):
    source_file = row['Filename']
    start = row['Start Time (s)']
    end = row['End Time (s)']
    species = row['Species eBird Code']
    duration = end - start

    source_path = os.path.join(AUDIO_PATH, source_file)

    # load only that segment
    y, sr = librosa.load(source_path, offset=start, duration=duration, sr=None)

    # build output filename
    clip_filename = f'{species}_{idx}.wav'
    output_path = os.path.join(OUTPUT_PATH, clip_filename)

    # save
    sf.write(output_path, y, sr)

    rows.append({
        'clip_filename': clip_filename,
        'species': species,
        'source_file': source_file,
        'path': output_path
    })

clips_df = pd.DataFrame(rows)
clips_df.to_csv('../data/clips_metadata.csv', index=False)
print(clips_df.shape)

  3%|▎         | 1429/49713 [00:06<04:01, 199.95it/s]C:\Users\Arup Sarkar\AppData\Local\Temp\ipykernel_18084\4095873198.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(source_path, offset=start, duration=duration, sr=None)
C:\Users\Arup Sarkar\.conda\envs\birdclef\lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  3%|▎         | 1445/49713 [00:06<03:41, 217.58it/s]


NoBackendError: 

In [8]:
print(source_path)
print(os.path.exists(source_path))
print(os.path.getsize(source_path) if os.path.exists(source_path) else 'N/A')

../soundscape_data\SSW_012_20170225_170013Z.flac
True
109265688


In [9]:
referenced_files = ann_filtered['Filename'].unique()
print(f'Files referenced in annotations: {len(referenced_files)}')

available_files = [f for f in referenced_files if os.path.exists(os.path.join(AUDIO_PATH, f))]
print(f'Files actually available: {len(available_files)}')

missing = set(referenced_files) - set(available_files)
print(f'Missing files: {len(missing)}')
print(list(missing)[:5])

Files referenced in annotations: 283
Files actually available: 283
Missing files: 0
[]
